# Inspeção do pipeline — uma imagem, do disco à segmentação

Conferência rápida do pipeline completo **antes** de um treino longo. Lê uma imagem pré-processada do disco (não precisa do
Cellpose nem de GPU) e:

1. mostra as entradas: imagem, ground truth, máscara e probabilidade do Cellpose e mapa de distância;
2. monta o pipeline do experimento-base — MarkerUNet + ScribblePrompt congelado no modo `positive` (só scribbles positivos,
   cortados em τ), com a entrada do ScribblePrompt em 256² — e confere que o gradiente chega à MarkerUNet, que o ScribblePrompt
   fica congelado e que os scribbles têm o formato esperado;
3. faz um **teste de sobreajuste numa imagem**: se o pipeline não consegue nem decorar uma imagem, não vai aprender com 37;
4. compara antes e depois do sobreajuste, com o Cellpose ao lado, e mostra marcadores, scribbles e segmentação.

O checkpoint do ScribblePrompt é baixado para `~/cell-fuzzy-models/scribbleprompt/`, fora do repositório. Em CPU, o notebook leva
poucos minutos.

## 0. Setup

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/Pedro-io/cell-fuzzy-seg.git"
BRANCH = "homolog"


def find_repo_root(start="."):
    p = os.path.abspath(start)
    prev = None
    while p != prev:
        if os.path.isdir(os.path.join(p, "src")) and os.path.exists(os.path.join(p, "pyproject.toml")):
            return p
        prev, p = p, os.path.dirname(p)
    return None


if "google.colab" in sys.modules:
    repo_root = os.path.join("/content", "cell-fuzzy-seg")
    if os.path.exists(repo_root):
        subprocess.run(["git", "-C", repo_root, "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", repo_root, "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", repo_root, "pull", "--ff-only", "origin", BRANCH], check=True)
    else:
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, repo_root], check=True)
else:
    repo_root = find_repo_root(os.getcwd())
    if repo_root is None:
        raise RuntimeError("Raiz do repositório não encontrada a partir do diretório atual.")

os.chdir(repo_root)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print("Repositório:", repo_root)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())

In [ ]:
if "google.colab" in sys.modules:
    get_ipython().system("python -m pip install -r requirements.txt -q")
else:
    print("Ambiente local: verifique se as dependências do requirements.txt estão instaladas.")

## 1. Imports e configuração

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from src.data.load.preprocessed_dataset import PreprocessedDataset, collate_samples, load_preprocessed
from src.evaluation.metrics import evaluate
from src.losses.loss_composer import LossComposer
from src.losses.terms import DiceTerm, TVTerm
from src.models.configs.marker_unet_config import UNET_CONFIG
from src.models.networks.final_segmentation.scribble_prompting_network import ScribblePromptingNetwork
from src.models.networks.marker_unet import MarkerUNet
from src.pipeline.steps.inference.frozen_segmentation_step import FrozenSegmentationStep
from src.pipeline.steps.inference.marker_step import MarkerStep
from src.pipeline.training_pipeline import TrainingPipeline
from src.training.trainer import Trainer
from src.training.training_loop import TrainingLoop

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
SP_INPUT_SIZE = (256, 256)   # entrada do ScribblePrompt
TAU = 0.5                    # limiar do canal positivo
OVERFIT_STEPS = 100          # passos do teste de sobreajuste
OVERFIT_LR = 1e-3            # maior que o dos experimentos, para o teste ser rápido
MODELS_DIR = os.path.expanduser(os.path.join("~", "cell-fuzzy-models"))

torch.manual_seed(SEED)
print("Device:", DEVICE)

## 2. Uma imagem pré-processada

In [ ]:
samples = load_preprocessed("train")
IMAGE_ID = sorted(samples)[0]
batch = collate_samples([PreprocessedDataset(samples, ids=[IMAGE_ID])[0]])
print("Imagem:", IMAGE_ID)
for key, value in batch.items():
    print(f"  {key:22s}", tuple(value.shape) if hasattr(value, "shape") else value, getattr(value, "dtype", ""))

s = samples[IMAGE_ID]
fig, axes = plt.subplots(1, 5, figsize=(25, 5))
for ax, (title, img, cmap) in zip(axes, [
    ("imagem", s["image"], None),
    ("ground truth", s["ground_truth"], "gray"),
    ("máscara do Cellpose", s["cellpose_segmentation"] > 0, "gray"),
    ("probabilidade do Cellpose", np.load(f"data_source/MoNuSegPreprocessed/train/cellpose_prob/{IMAGE_ID}.npy"), "magma"),
    ("mapa de distância (0 no centro de cada núcleo)", s["distance_map"], "viridis"),
]):
    ax.imshow(img, cmap=cmap)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Pipeline do experimento-base e conferências estruturais

In [ ]:
checkpoint = ScribblePromptingNetwork.download_checkpoint(os.path.join(MODELS_DIR, "scribbleprompt"))
final_net = ScribblePromptingNetwork(
    checkpoint=checkpoint, device=DEVICE, scribble_mode="positive",
    positive_threshold=TAU, input_size=SP_INPUT_SIZE,
)
marker_net = MarkerUNet(config=UNET_CONFIG)
pipeline = TrainingPipeline([
    MarkerStep(model=marker_net, differentiable=True, device=DEVICE),
    FrozenSegmentationStep(final_network=final_net, device=DEVICE),
])
composer = LossComposer([DiceTerm(), TVTerm(weight=0.001)])

# Um forward + backward: o gradiente tem de chegar à MarkerUNet e o ScribblePrompt não pode ter parâmetros treináveis.
data = pipeline.run({k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in batch.items()}, verbose=False)
loss, log = composer(data["segmentation"], data["distance_map"], data["ground_truth"], markers=data["markers"])
loss.backward()
grad = sum(p.grad.abs().sum().item() for p in marker_net.parameters() if p.grad is not None)
treinaveis_sp = sum(p.numel() for p in final_net.parameters() if p.requires_grad)
scribbles = final_net._prepare_scribbles(data["markers"].detach(), torch.device(DEVICE))
m = data["markers"].detach()

print(f"markers {tuple(m.shape)} em [{m.min():.3f}, {m.max():.3f}] | segmentação {tuple(data['segmentation'].shape)}")
print(f"perda {loss.item():.4f}", {k: round(v.item(), 4) for k, v in log.items()})
print(f"soma |grad| na MarkerUNet: {grad:.3e} | parâmetros treináveis no ScribblePrompt: {treinaveis_sp}")
assert grad > 0, "o gradiente não chegou à MarkerUNet"
assert treinaveis_sp == 0, "o ScribblePrompt deveria estar congelado"
assert torch.all(scribbles[:, 1] == 0), "o canal negativo deveria ser zero no modo positive"
assert torch.all(scribbles[:, 0][m[:, 0] <= TAU] == 0), "o positivo deveria ser zero abaixo de τ"
marker_net.zero_grad()
print("Conferências estruturais: ok")

## 4. Antes do treino

In [ ]:
rows_before = evaluate(pipeline, [batch], device=DEVICE)  # põe as redes em modo de avaliação
with torch.no_grad():
    before = pipeline.run({k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in batch.items()}, verbose=False)
markers_before = before["markers"][0, 0].cpu().numpy()
r = rows_before[0]
print(f"antes: Dice {r['dice']:.3f} | massa {r['mass_ratio']:.2f} | marcador cobre {100 * r['marker_fraction']:.1f}% "
      f"| Cellpose: Dice {r['cellpose_dice']:.3f}")

## 5. Teste de sobreajuste numa imagem

Treina só com esta imagem, sem aumentação, por `OVERFIT_STEPS` passos. É um teste de sanidade: a perda tem de cair e o Dice nesta
imagem tem de subir. Não diz nada sobre generalização.

In [ ]:
optimizer = torch.optim.Adam(marker_net.parameters(), lr=OVERFIT_LR)
trainer = Trainer(pipeline, composer, optimizer, device=DEVICE, grad_clip=1.0)
history = TrainingLoop(trainer, train_loader=[batch], num_epochs=OVERFIT_STEPS, log_every=10).run()

plt.figure(figsize=(6, 3))
plt.plot(history["train_loss"])
plt.xlabel("passo")
plt.ylabel("perda (Dice + TV)")
plt.title("Sobreajuste numa imagem")
plt.show()
print(f"perda: {history['train_loss'][0]:.4f} → {history['train_loss'][-1]:.4f}")

## 6. Depois do sobreajuste

In [ ]:
rows_after = evaluate(pipeline, [batch], device=DEVICE)
with torch.no_grad():
    after = pipeline.run({k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in batch.items()}, verbose=False)
a = rows_after[0]
print(f"depois: Dice {a['dice']:.3f} | massa {a['mass_ratio']:.2f} | marcador cobre {100 * a['marker_fraction']:.1f}% "
      f"| Cellpose: Dice {a['cellpose_dice']:.3f}")

markers_after = after["markers"][0, 0].cpu().numpy()
scribbles_after = final_net._prepare_scribbles(after["markers"], torch.device(DEVICE))[0, 0].cpu().numpy()
fig, axes = plt.subplots(1, 6, figsize=(30, 5))
for ax, (title, img, cmap) in zip(axes, [
    ("ground truth", s["ground_truth"], "gray"),
    ("Cellpose", s["cellpose_segmentation"] > 0, "gray"),
    ("marcadores antes", markers_before, "magma"),
    ("marcadores depois", markers_after, "magma"),
    (f"scribble positivo depois (corte em τ = {TAU})", scribbles_after, "magma"),
    ("segmentação depois (≥ 0,5)", after["segmentation"][0, 0].cpu().numpy() >= 0.5, "gray"),
]):
    ax.imshow(img, cmap=cmap, vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()